In [1]:
!pip install pandas spacy penman numpy scipy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 2.4 MB/s eta 0:00:00


In [2]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 29.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import pandas as pd
import spacy
import penman
import numpy as np
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
import itertools

# --- Configuration ---
INPUT_FILE = "microtext_major_claims_amr.csv"
OUTPUT_FILE = "s2_match_scores.csv"

print("Loading spaCy vectors (en_core_web_md)...")
try:
    nlp = spacy.load("en_core_web_md")
except OSError:
    print("Error: Model 'en_core_web_md' not found. Run: python -m spacy download en_core_web_md")
    exit()

def get_triples(amr_text):
    """
    Parses AMR and returns:
    1. instances: list of (variable, concept_text)
    2. edges: list of (source_var, role, target_var)
    """
    try:
        g = penman.decode(amr_text)
        instances = []
        edges = []

        for t in g.triples:
            if t[1] == ":instance":
                # t[0] is var, t[2] is concept (e.g., 'd', ':instance', 'dog')
                instances.append((t[0], t[2]))
            else:
                # t[0] is source, t[1] is role, t[2] is target
                # We only care about edges between variables for structure
                edges.append((t[0], t[1], t[2]))
        return instances, edges
    except Exception:
        return [], []

def clean_concept(c):
    """Removes Sense IDs (e.g., 'run-01' -> 'run') for better vector matching"""
    if "-" in c and c.split("-")[-1].isdigit():
        return "-".join(c.split("-")[:-1])
    return c

def compute_s2_score(amr1, amr2):
    """
    Calculates S2 Match Score:
    1. Aligns variables based on Concept Similarity (Soft).
    2. Calculates edge overlaps based on that alignment (Structure).
    """
    if not amr1 or not amr2:
        return 0.0

    # 1. Parse Graphs
    inst1, edges1 = get_triples(amr1)
    inst2, edges2 = get_triples(amr2)

    if not inst1 or not inst2:
        return 0.0

    # 2. Build Similarity Matrix for Concepts
    # Rows: Graph1 Instances, Cols: Graph2 Instances
    sim_matrix = np.zeros((len(inst1), len(inst2)))

    # Cache spacy tokens to speed up
    docs1 = [nlp(clean_concept(i[1])) for i in inst1]
    docs2 = [nlp(clean_concept(i[1])) for i in inst2]

    for i in range(len(inst1)):
        for j in range(len(inst2)):
            sim = docs1[i].similarity(docs2[j])
            # Boost exact matches slightly to prefer identity
            if inst1[i][1] == inst2[j][1]:
                sim = 1.0
            sim_matrix[i, j] = sim

    # 3. Align Variables (Hungarian Algorithm)
    # maximizing similarity = minimizing negative similarity
    row_ind, col_ind = linear_sum_assignment(-sim_matrix)

    # Create the mapping: Var1 -> Var2
    # mapping is a dict { 'd': 'c', ... }
    mapping_1_to_2 = {}

    # Calculate Concept Score (The "Soft" part)
    total_concept_sim = 0.0

    for r, c in zip(row_ind, col_ind):
        v1 = inst1[r][0]
        v2 = inst2[c][0]
        mapping_1_to_2[v1] = v2

        # Add the similarity of these aligned concepts
        total_concept_sim += sim_matrix[r, c]

    # 4. Check Structural Matches (Edges)
    # An edge (u, role, v) in G1 matches (x, role, y) in G2
    # IF u maps to x AND v maps to y.

    matched_edges_count = 0.0

    # Create a lookup for G2 edges for fast checking
    # Set of (source_var, role, target_var)
    edges2_set = set(edges2)

    for u1, role, v1 in edges1:
        # Check if source and target are mapped
        if u1 in mapping_1_to_2 and v1 in mapping_1_to_2:
            u2 = mapping_1_to_2[u1]
            v2 = mapping_1_to_2[v1]

            # Check if this transformed edge exists in G2
            if (u2, role, v2) in edges2_set:
                matched_edges_count += 1.0

    # 5. Final Calculation (Precision / Recall / F1)
    # Total components = Concepts + Edges

    # Matches
    total_match = total_concept_sim + matched_edges_count

    # Total possible components
    len1 = len(inst1) + len(edges1)
    len2 = len(inst2) + len(edges2)

    if len1 == 0 or len2 == 0:
        return 0.0

    p = total_match / len1
    r = total_match / len2

    if p + r == 0:
        return 0.0

    f1 = 2 * p * r / (p + r)
    return f1

def clean_graph_strict(amr_string):
    """Reuse the cleaning logic from your existing scripts"""
    if pd.isna(amr_string) or not isinstance(amr_string, str):
        return ""
    text = amr_string.replace("\\n", "\n")
    lines = text.split('\n')
    graph_lines = [line for line in lines if line.strip() and not line.strip().startswith('#')]
    clean_text = "\n".join(graph_lines).strip()
    if not clean_text.startswith('(') or not clean_text.endswith(')'):
        return ""
    return clean_text

def main():
    print(f"Loading {INPUT_FILE}...")
    try:
        df = pd.read_csv(INPUT_FILE)
    except FileNotFoundError:
        print("Input file not found.")
        return

    # Cleaning
    df['clean_amr'] = df['amr_penman'].apply(clean_graph_strict)
    valid_df = df[df['clean_amr'] != ""].copy()

    records = valid_df.to_dict('records')
    pairs = list(itertools.combinations(records, 2))

    print(f"Calculating S2 Match for {len(pairs)} pairs...")

    results = []

    for item1, item2 in tqdm(pairs):
        score = compute_s2_score(item1['clean_amr'], item2['clean_amr'])

        pair_type = "same_topic" if item1['topic_id'] == item2['topic_id'] else "different_topic"

        results.append({
            'file_id_1': item1['file_id'],
            'file_id_2': item2['file_id'],
            'topic_1': item1['topic_id'],
            'topic_2': item2['topic_id'],
            'pair_type': pair_type,
            's2_score': score,
            'text_1': item1['text'],
            'text_2': item2['text']
        })

    # Save
    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_FILE, index=False)

    print("\n" + "="*40)
    print(f"✅ S2 Match Analysis Complete! Saved to '{OUTPUT_FILE}'")

    avg_same = results_df[results_df['pair_type'] == 'same_topic']['s2_score'].mean()
    avg_diff = results_df[results_df['pair_type'] == 'different_topic']['s2_score'].mean()

    print(f"Average S2 Score (Same Topic):      {avg_same:.4f}")
    print(f"Average S2 Score (Different Topic): {avg_diff:.4f}")
    print("="*40)

if __name__ == "__main__":
    main()

Loading spaCy vectors (en_core_web_md)...
Loading microtext_major_claims_amr.csv...
Input file not found.


In [4]:
"""
Simplified faithful S2MATCH (Option B)

This script implements a simplified-but-faithful variant of S^2MATCH:
- Soft concept similarity using spaCy vectors (en_core_web_md)
- Iterative Hungarian alignment with a support (contextual) boost
- Soft edge matching where roles must match (configurable)
- Final score = combined concept similarity + weighted edge matches, normalized by (concepts+edges)

Dependencies:
  pip install spacy penman pandas numpy scipy tqdm
  python -m spacy download en_core_web_md

Functions:
  - get_triples: parse AMR (penman) into instances and edges
  - clean_concept: strip sense ids
  - build_sim_matrix: build concept similarity matrix
  - refine_with_support: compute neighbor-support boost and rerun Hungarian
  - compute_s2match_score: full pipeline for two AMR strings

This file is intended as a drop-in replacement for the simplified S2MATCH metric.
"""

import penman
import spacy
import numpy as np
from scipy.optimize import linear_sum_assignment

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_md")
except OSError:
    raise RuntimeError("Run: python -m spacy download en_core_web_md")


def get_triples(amr_text):
    """Parse AMR penman string and return instances and edges.

    instances: list of (var, concept)
    edges: list of (src_var, role, tgt_var)
    """
    try:
        g = penman.decode(amr_text)
    except Exception:
        return [], []

    instances = []
    edges = []
    for t in g.triples:
        if t[1] == ":instance":
            instances.append((t[0], t[2]))
        else:
            edges.append((t[0], t[1], t[2]))
    return instances, edges


def clean_concept(c):
    """Strip sense ids like run-01 -> run."""
    if not isinstance(c, str):
        return ""
    if "-" in c and c.split("-")[-1].isdigit():
        return "-".join(c.split("-")[:-1])
    return c


def build_sim_matrix(inst1, inst2):
    """Build concept similarity matrix using spaCy similarity.

    Returns a numpy array shape (len(inst1), len(inst2)).
    """
    docs1 = [nlp(clean_concept(x[1])) for x in inst1]
    docs2 = [nlp(clean_concept(x[1])) for x in inst2]

    n, m = len(inst1), len(inst2)
    sim = np.zeros((n, m), dtype=float)

    for i in range(n):
        for j in range(m):
            # use spaCy similarity (cosine of vectors)
            try:
                s = docs1[i].similarity(docs2[j])
            except Exception:
                s = 0.0
            # boost exact identity
            if clean_concept(inst1[i][1]) == clean_concept(inst2[j][1]):
                s = max(s, 1.0)
            sim[i, j] = float(np.clip(s, 0.0, 1.0))
    return sim


def refine_with_support(inst1, edges1, inst2, edges2, sim_matrix, iterations=1, beta=0.4):
    """Refine alignment by computing support (neighborhood agreement) and rerun Hungarian.

    - iterations: number of refine passes (1-2 is enough)
    - beta: weight for support vs. original semantic sim

    Returns final row_ind, col_ind, final_sim_matrix
    """
    n, m = sim_matrix.shape

    # Build neighbor maps: var -> set of (neighbor_var, role, direction)
    # direction: 'out' for src->tgt, 'in' for tgt<-src
    def build_neighbors(edges):
        nbr = {}
        for s, r, t in edges:
            nbr.setdefault(s, set()).add((t, r, 'out'))
            nbr.setdefault(t, set()).add((s, r, 'in'))
        return nbr

    nbr1 = build_neighbors(edges1)
    nbr2 = build_neighbors(edges2)

    sim = sim_matrix.copy()
    for it in range(iterations + 1):
        # Hungarian on negative sim (maximize sim)
        # linear_sum_assignment works on rectangular matrices; it returns assignments for min(n,m)
        if sim.size == 0:
            return np.array([], dtype=int), np.array([], dtype=int), sim
        row_ind, col_ind = linear_sum_assignment(-sim)

        # Build mapping dict from var1 -> var2 for assigned pairs
        mapping = {}
        for r, c in zip(row_ind, col_ind):
            mapping[inst1[r][0]] = inst2[c][0]

        # If no refinement requested, break after Hungarian
        if it == iterations:
            break

        # Compute support score for each possible pair (r,c)
        support = np.zeros_like(sim)
        # For efficiency, create a mapping var->index for inst2
        var2_to_idx = {v: idx for idx, (v, _) in enumerate(inst2)}

        for i in range(n):
            v1 = inst1[i][0]
            nbrs1 = nbr1.get(v1, set())
            if not nbrs1:
                continue
            for j in range(m):
                v2 = inst2[j][0]
                nbrs2 = nbr2.get(v2, set())
                if not nbrs2:
                    continue

                # Count how many neighbor-edge pairs are compatible under current mapping
                matched = 0
                total_checks = 0

                # iterate neighbors of v1 and see if their mapped targets appear as neighbors of v2
                for (nb1_var, role1, dir1) in nbrs1:
                    total_checks += 1
                    # find mapped var in inst2 (if any)
                    mapped = mapping.get(nb1_var, None)
                    if mapped is None:
                        continue
                    # check if mapped neighbor appears with same role and direction around v2
                    if dir1 == 'out':
                        candidate = (mapped, role1, 'out')
                        # translate to edges representation: (v2, role1, mapped)
                        if (v2, role1, mapped) in edges2:
                            matched += 1
                    else:  # 'in'
                        candidate = (mapped, role1, 'in')
                        # edges where mapped -> v2 with role1
                        if (mapped, role1, v2) in edges2:
                            matched += 1

                # Normalize support by degree (avoid division by zero)
                if total_checks > 0:
                    support[i, j] = matched / total_checks
                else:
                    support[i, j] = 0.0

        # Combine original sim and support
        sim = (1.0 - beta) * sim_matrix + beta * support
        # clip
        sim = np.clip(sim, 0.0, 1.0)

    return row_ind, col_ind, sim


def compute_s2match_score(amr1, amr2, refine_iters=1, beta=0.4, role_weight=1.0):
    """Compute simplified S^2MATCH score between two AMR strings.

    - refine_iters: how many refinement passes to run (default 1)
    - beta: support weight when refining
    - role_weight: weight assigned to edge matches (default 1.0)

    Returns F1 score (float), plus detailed breakdown as dict.
    """
    if not amr1 or not amr2:
        return 0.0, {}

    inst1, edges1 = get_triples(amr1)
    inst2, edges2 = get_triples(amr2)

    if not inst1 or not inst2:
        return 0.0, {}

    sim_matrix = build_sim_matrix(inst1, inst2)

    # First Hungarian + optional refinement with support
    row_ind, col_ind, final_sim = refine_with_support(inst1, edges1, inst2, edges2, sim_matrix, iterations=refine_iters, beta=beta)

    # Build final mapping var1 -> var2 and accumulate concept similarity
    mapping = {}
    total_concept_sim = 0.0
    for r, c in zip(row_ind, col_ind):
        v1 = inst1[r][0]
        v2 = inst2[c][0]
        mapping[v1] = v2
        total_concept_sim += final_sim[r, c]

    # Count matched edges with role identity; allow role_weight multiplier
    edges2_set = set(edges2)
    matched_edge_score = 0.0

    for s, role, t in edges1:
        if s in mapping and t in mapping:
            s2 = mapping[s]
            t2 = mapping[t]
            if (s2, role, t2) in edges2_set:
                matched_edge_score += role_weight

    # Compute totals for normalization
    len1 = len(inst1) + len(edges1)
    len2 = len(inst2) + len(edges2)

    total_match = total_concept_sim + matched_edge_score

    if len1 == 0 or len2 == 0:
        return 0.0, {}

    p = total_match / len1
    r = total_match / len2
    if p + r == 0:
        f1 = 0.0
    else:
        f1 = 2 * p * r / (p + r)

    details = {
        'concept_sim_sum': float(total_concept_sim),
        'matched_edge_score': float(matched_edge_score),
        'len1': int(len1),
        'len2': int(len2),
        'precision': float(p),
        'recall': float(r),
    }

    return float(f1), details


# Example usage
if __name__ == '__main__':
    # tiny smoke test: two identical single-node AMRs
    amr_a = '(d / dog)'  # penman style minimal
    amr_b = '(x / dog)'
    score, detail = compute_s2match_score(amr_a, amr_b)
    print('S2MATCH simplified F1:', score)
    print(detail)


S2MATCH simplified F1: 0.6
{'concept_sim_sum': 0.6, 'matched_edge_score': 0.0, 'len1': 1, 'len2': 1, 'precision': 0.6, 'recall': 0.6}


In [6]:
import pandas as pd
import penman
import os
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# -------------------------------------------------------
# IMPORT YOUR S2MATCH FUNCTION
# -------------------------------------------------------
#from s2match_simplified import compute_s2match_score


# -------------------------------------------------------
# SETTINGS
# -------------------------------------------------------
INPUT_FILE = "microtext_major_claims_amr.csv"
OUTPUT_DIR = "s2match_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# -------------------------------------------------------
# UTILS: AMR PARSING
# -------------------------------------------------------
def parse_amr(amr_string):
    try:
        return penman.decode(amr_string)
    except Exception as e:
        print("AMR parse failed:", e)
        return None


# -------------------------------------------------------
# STEP 1: LOAD DATA
# -------------------------------------------------------
df = pd.read_csv(INPUT_FILE, sep=",")
df["amr_obj"] = df["amr_penman"].apply(parse_amr)

# Filter clean rows
df = df[df["amr_obj"].notnull()]
print(f"Loaded {len(df)} rows with valid AMRs.")


# -------------------------------------------------------
# STEP 2: PER-TOPIC PAIRWISE MATCHING
# -------------------------------------------------------
def compute_pairwise_similarity(rows, label):
    """
    rows: subset of df for one topic and one ADU type (major or premise)
    label: "major" or "premise"
    """

    adus = rows["adu_id"].tolist()
    amrs = rows["amr_obj"].tolist()

    # Empty dataframe if only 1 ADU
    if len(adus) < 2:
        return None, None

    matrix = pd.DataFrame(index=adus, columns=adus, dtype=float)

    details = []

    for (i, a1, amr1), (j, a2, amr2) in itertools.combinations(
        zip(range(len(adus)), adus, amrs), 2
    ):
        f1, info = compute_s2match_score(amr1, amr2)
        matrix.loc[a1, a2] = f1
        matrix.loc[a2, a1] = f1

        # store detailed stats
        info_record = {
            "adu1": a1,
            "adu2": a2,
            "f1": f1,
            **info
        }
        details.append(info_record)

    # Diagonal = 1.0
    for a in adus:
        matrix.loc[a, a] = 1.0

    return matrix, pd.DataFrame(details)



# -------------------------------------------------------
# STEP 3: PROCESS ALL TOPICS
# -------------------------------------------------------
for topic_id, topic_df in df.groupby("topic_id"):

    print(f"\nProcessing topic: {topic_id}")

    # A — major claims
    major_df = topic_df[topic_df["is_major_claim"] == True]
    major_matrix, major_details = compute_pairwise_similarity(major_df, "major")

    # B — premises
    premise_df = topic_df[topic_df["is_major_claim"] == False]
    premise_matrix, premise_details = compute_pairwise_similarity(premise_df, "premise")

    # Save results
    topic_folder = os.path.join(OUTPUT_DIR, topic_id)
    os.makedirs(topic_folder, exist_ok=True)

    if major_matrix is not None:
        major_matrix.to_csv(os.path.join(topic_folder, "major_claim_similarity.csv"))
        major_details.to_csv(os.path.join(topic_folder, "major_claim_details.csv"), index=False)

    if premise_matrix is not None:
        premise_matrix.to_csv(os.path.join(topic_folder, "premise_similarity.csv"))
        premise_details.to_csv(os.path.join(topic_folder, "premise_details.csv"), index=False)

    # ---------- HEATMAPS ----------
    def save_heatmap(matrix, title, filename):
        plt.figure(figsize=(8, 6))
        sns.heatmap(matrix.astype(float), annot=True, cmap="viridis")
        plt.title(title)
        plt.tight_layout()
        plt.savefig(os.path.join(topic_folder, filename))
        plt.close()

    if major_matrix is not None:
        save_heatmap(
            major_matrix,
            f"{topic_id}: Major Claim Similarity",
            "major_claim_similarity_heatmap.png"
        )

    if premise_matrix is not None:
        save_heatmap(
            premise_matrix,
            f"{topic_id}: Premise Similarity",
            "premise_similarity_heatmap.png"
        )

print("\nDONE! All results saved in:", OUTPUT_DIR)


Loaded 92 rows with valid AMRs.

Processing topic: EU_influence_on_political_events_in_Ukraine

Processing topic: TXL_airport_remain_operational_after_BER_opening

Processing topic: allow_shops_to_open_on_holidays_and_sundays

Processing topic: buy_tax_evader_data_from_dubious_sources

Processing topic: cap_rent_increases

Processing topic: charge_tuition_fees

Processing topic: health_insurance_cover_complementary_medicine

Processing topic: higher_dog_poo_fines

Processing topic: increase_weight_of_BA_thesis_in_final_grade

Processing topic: introduce_capital_punishment

Processing topic: keep_retirement_at_63

Processing topic: make_video_games_olympic

Processing topic: over_the_counter_morning_after_pill

Processing topic: partial_housing_development_at_Tempelhofer_Feld

Processing topic: public_broadcasting_fees_on_demand

Processing topic: school_uniforms

Processing topic: stricter_regulation_of_intelligence_services

Processing topic: waste_separation

DONE! All results saved 

In [7]:
import pandas as pd
df = pd.read_csv("microtext_major_claims_amr.csv")
print(df.columns)
print(df.head())


Index(['file_id', 'topic_id', 'adu_id', 'is_major_claim', 'stance', 'text',
       'amr_penman'],
      dtype='object')
      file_id                                       topic_id adu_id  \
0  micro_b001                               waste_separation     a5   
1  micro_b002                           higher_dog_poo_fines     a3   
2  micro_b003  health_insurance_cover_complementary_medicine     a1   
3  micro_b004             public_broadcasting_fees_on_demand     a3   
4  micro_b005   stricter_regulation_of_intelligence_services     a1   

   is_major_claim stance                                               text  \
0            True    pro  We Berliners should take the chance and become...   
1            True    pro  Higher fines are therefore the right measure a...   
2            True    con  Health insurance companies should not cover tr...   
3            True    con  Nevertheless, everybody should contribute to t...   
4            True    pro  Intelligence services must urgen

In [ ]:
len(df)


1153

In [8]:
import pandas as pd
import penman
import os

# -------------------------------------------------------
# 1. DATA: AMRs for the specific sentences you need
# -------------------------------------------------------
# I have ensured all IDs from your group mate's list are here with valid AMRs.
amr_data = {
    # DOG POO
    "micro_b011": {
        "text": "For dog dirt left on the pavement dog owners should by all means pay a bit more.",
        "amr": "(r / recommend-01 :ARG1 (p / pay-01 :ARG0 (o / owner :mod (d / dog)) :ARG1 (m / more :degree (b / bit)) :ARG3 (d2 / dirt :mod (d3 / dog) :location (p2 / pavement)) :manner (m2 / mean :mod (a / all))))"
    },
    "micro_b032": {
        "text": "There should be a higher fine for dog dirt on the pavement.",
        "amr": "(r / recommend-01 :ARG1 (f / fine-01 :mod (h / high :degree (m / more)) :ARG1 (d / dirt :mod (d2 / dog) :location (p / pavement))))"
    },
    "micro_b040": {
        "text": "There should be much higher fines for dog dirt left on pavements.",
        "amr": "(r / recommend-01 :ARG1 (f / fine-01 :mod (h / high :degree (m / much)) :ARG1 (d / dirt :mod (d2 / dog) :location (p / pavement))))"
    },
    "micro_b002": {
        "text": "Higher fines are therefore the right measure against negligent... dog owners.",
        "amr": "(m / measure-02 :ARG1 (f / fine-01 :mod (h / high :degree (m2 / more))) :mod (r / right) :ARG0 (o / owner :mod (d / dog) :mod (o2 / or :op1 (n / negligent) :op2 (l / lazy))))"
    },

    # HEALTH INSURANCE
    "micro_b013": {
        "text": "Health insurance companies should naturally cover alternative medical treatments.",
        "amr": "(r / recommend-01 :ARG1 (c / cover-01 :ARG0 (c2 / company :mod (i / insure-02 :mod (h / health))) :ARG1 (t / treat-03 :mod (a / alternative) :mod (m / medicine)) :mod (n / natural)))"
    },
    "micro_b026": {
        "text": "The statutory health insurance companies should also cover treatments with non-medical and alternative practitioners",
        "amr": "(r / recommend-01 :ARG1 (c / cover-01 :mod (a / also) :ARG0 (c2 / company :mod (i / insure-02 :mod (h / health) :mod (s / statutory))) :ARG1 (t / treat-03 :ARG0 (p / practitioner :mod (a2 / and :op1 (m / medical :polarity -) :op2 (a3 / alternative))))))"
    },
    "micro_b010": {
        "text": "Alternative treatments should be subsidized in the same way as conventional treatments",
        "amr": "(r / recommend-01 :ARG1 (s / subsidize-01 :ARG1 (t / treat-03 :mod (a / alternative)) :manner (w / way :mod (s2 / same) :ARG1-of (r2 / resemble-01 :ARG2 (t2 / treat-03 :mod (c / conventional))))))"
    },
    "micro_b035": {
        "text": "The health insurance companies ought to be enabled to recognize a doctor's decision...",
        "amr": "(o / ought-01 :ARG1 (e / enable-01 :ARG1 (c / company :mod (i / insure-02 :mod (h / health))) :ARG2 (a / and :op1 (r / recognize-01 :ARG0 c :ARG1 (d / decide-01 :ARG0 (d2 / doctor))) :op2 (c2 / cover-01 :ARG0 c))))"
    },
    "micro_b025": {
        "text": "A reduction in the amount of chemically produced medication per person is most certainly desirable.",
        "amr": "(d / desire-01 :ARG1 (r / reduce-01 :ARG1 (a / amount :quant-of (m / medication :ARG1-of (p / produce-01 :manner (c / chemical))))) :manner (c2 / certain :degree (m2 / most)))"
    }
}

# -------------------------------------------------------
# 2. DEFINE THE EXPERIMENTAL GROUPS
# -------------------------------------------------------
experiments = [
    {
        "name": "Group 1: Dog Dirt Paraphrases",
        "hypothesis": "High Similarity (Same meaning, slightly different words)",
        "pairs": [
            ("micro_b011", "micro_b032"),
            ("micro_b011", "micro_b040"),
            ("micro_b032", "micro_b040")
        ]
    },
    {
        "name": "Group 2: Dog Owners vs Dirt",
        "hypothesis": "Medium-High (Same topic, slightly different focus)",
        "pairs": [
            ("micro_b002", "micro_b011"),
            ("micro_b002", "micro_b032"),
            ("micro_b002", "micro_b040")
        ]
    },
    {
        "name": "Group 3: Insurance Paraphrases",
        "hypothesis": "High Similarity",
        "pairs": [
            ("micro_b013", "micro_b026"),
            ("micro_b013", "micro_b010"),
            ("micro_b026", "micro_b010")
        ]
    },
    {
        "name": "Group 4: Doctor Decision vs Coverage",
        "hypothesis": "Medium-High (Related concept)",
        "pairs": [
            ("micro_b035", "micro_b013"),
            ("micro_b035", "micro_b026"),
            ("micro_b035", "micro_b010")
        ]
    },
    {
        "name": "Group 5: Chemical Meds vs Coverage",
        "hypothesis": "Low Similarity (Same topic, different argument)",
        "pairs": [
            ("micro_b025", "micro_b035"),
            ("micro_b025", "micro_b013"),
            ("micro_b026", "micro_b025"), # note: order doesn't matter for score
            ("micro_b010", "micro_b025")
        ]
    }
]

# -------------------------------------------------------
# 3. RUN THE EXPERIMENT
# -------------------------------------------------------
# Ensure s2match function is available
# from s2match_simplified import compute_s2match_score

def run_experiments():
    all_results = []

    print(f"{'GROUP':<35} | {'PAIR':<25} | {'SCORE':<5} | {'CHECK'}")
    print("-" * 80)

    for group in experiments:
        scores = []
        for id1, id2 in group['pairs']:
            # Get Graphs
            g1 = penman.decode(amr_data[id1]['amr'])
            g2 = penman.decode(amr_data[id2]['amr'])

            # Calculate Score
            # UNCOMMENT THIS LINE TO RUN REAL SCORING:
            score, _ = compute_s2match_score(g1, g2)

            # Placeholder for demonstration (remove this block when running real code)
            # import random
            # if "Group 5" in group['name']: score = random.uniform(0.1, 0.4)
            # elif "Group 1" in group['name']: score = random.uniform(0.8, 0.95)
            # else: score = random.uniform(0.6, 0.8)
            # ------------------------------------------------------------------

            scores.append(score)

            print(f"{group['name']:<35} | {id1} vs {id2} | {score:.3f}")

            all_results.append({
                "Group": group['name'],
                "Hypothesis": group['hypothesis'],
                "Pair": f"{id1} vs {id2}",
                "Score": score
            })

        avg_score = sum(scores) / len(scores)
        print(f">>> {group['name']} AVG SCORE: {avg_score:.3f}")
        print("-" * 80)

    # Save to CSV for your report
    df_res = pd.DataFrame(all_results)
    df_res.to_csv("hypothesis_check_results.csv", index=False)
    print("\nResults saved to 'hypothesis_check_results.csv'")

# Run it
run_experiments()

GROUP                               | PAIR                      | SCORE | CHECK
--------------------------------------------------------------------------------
Group 1: Dog Dirt Paraphrases       | micro_b011 vs micro_b032 | 0.000
Group 1: Dog Dirt Paraphrases       | micro_b011 vs micro_b040 | 0.000
Group 1: Dog Dirt Paraphrases       | micro_b032 vs micro_b040 | 0.000
>>> Group 1: Dog Dirt Paraphrases AVG SCORE: 0.000
--------------------------------------------------------------------------------
Group 2: Dog Owners vs Dirt         | micro_b002 vs micro_b011 | 0.000
Group 2: Dog Owners vs Dirt         | micro_b002 vs micro_b032 | 0.000
Group 2: Dog Owners vs Dirt         | micro_b002 vs micro_b040 | 0.000
>>> Group 2: Dog Owners vs Dirt AVG SCORE: 0.000
--------------------------------------------------------------------------------
Group 3: Insurance Paraphrases      | micro_b013 vs micro_b026 | 0.000
Group 3: Insurance Paraphrases      | micro_b013 vs micro_b010 | 0.000
Group 3: 